In [ ]:
import os
import pickle
import numpy as np
import pandas as pd

from tqdm import tqdm
from matplotlib import pyplot as plt

import tensorflow as tf
import keras.backend as K

from tensorflow.keras.models import Model, Sequential, load_model
from tensorflow.keras.applications import MobileNetV3Large
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D, Activation, Flatten # usually you end on a Dense layer, Flatten before the dense layer
from tensorflow.keras.callbacks import EarlyStopping, History, ModelCheckpoint

# XGBoost
from xgboost import XGBClassifier

# SVM
import joblib
from sklearn.svm import SVC

# XGBoost + SVM
from sklearn.metrics import accuracy_score

In [ ]:
def f1_score(y_true, y_pred): #taken from old keras source code
    true_positives = K.sum(K.round(K.clip(y_true * y_pred, 0, 1)))
    possible_positives = K.sum(K.round(K.clip(y_true, 0, 1)))
    predicted_positives = K.sum(K.round(K.clip(y_pred, 0, 1)))
    precision = true_positives / (predicted_positives + K.epsilon())
    recall = true_positives / (possible_positives + K.epsilon())
    f1_val = 2*(precision*recall)/(precision+recall+K.epsilon())
    return f1_val

DISPOSITIVOS = ['dish washer', 'kettle', 'microwave', 'washing machine', 'fridge']
#DISPOSITIVOS = ['kettle']
IMAGENS = ['rp', 'gadf','gasf','mtf','tbg']
#IMAGENS = ['tbg']

In [ ]:
def dividir_dados():
    for disp in DISPOSITIVOS:

        y = pickle.load(open(f"pickle_data_tbg/y_({disp}).pickle","rb"))

        y_train = y[:int(len(y)*0.6)]
        y_val = y[len(y_train):len(y_train) + int(len(y)*0.2)]
        y_test = y[(len(y_train) + len(y_val)):]

        pickle_out = open(f"pickle_data_tbg/y_train({disp}).pickle", "wb")
        pickle.dump(y_train, pickle_out)
        pickle_out.close()

        pickle_out = open(f"pickle_data_tbg/y_val({disp}).pickle", "wb")
        pickle.dump(y_val, pickle_out)
        pickle_out.close()

        pickle_out = open(f"pickle_data_tbg/y_test({disp}).pickle", "wb")
        pickle.dump(y_test, pickle_out)
        pickle_out.close()

        for img in IMAGENS:

            X = pickle.load(open(f"pickle_data_tbg/X_{img}_({disp}).pickle","rb"))

            X_train = X[:int(len(X)*0.6)]
            X_val = X[len(X_train):len(X_train) + int(len(X)*0.2)]
            X_test = X[(len(X_train) + len(X_val)):]

            pickle_out = open(f"pickle_data_tbg/X_{img}_train({disp}).pickle", "wb")
            pickle.dump(X_train, pickle_out)
            pickle_out.close()

            pickle_out = open(f"pickle_data_tbg/X_{img}_val({disp}).pickle", "wb")
            pickle.dump(X_val, pickle_out)
            pickle_out.close()

            pickle_out = open(f"pickle_data_tbg/X_{img}_test({disp}).pickle", "wb")
            pickle.dump(X_test, pickle_out)
            pickle_out.close()

    print("Divisão concluida com sucesso.")

def dividir_dados_val_only():
    for disp in DISPOSITIVOS:

        y = pickle.load(open(f"pickle_data/y_train({disp}).pickle","rb"))

        y_train = y[:int(len(y)*0.8)]
        y_val = y[len(y_train):]

        pickle_out = open(f"pickle_data/y_train_split({disp}).pickle", "wb")
        pickle.dump(y_train, pickle_out)
        pickle_out.close()

        pickle_out = open(f"pickle_data/y_val({disp}).pickle", "wb")
        pickle.dump(y_val, pickle_out)
        pickle_out.close()

        for img in IMAGENS:

            X = pickle.load(open(f"pickle_data/X_{img}_train({disp}).pickle","rb"))

            X_train = X[:int(len(X)*0.8)]
            X_val = X[len(X_train):]

            pickle_out = open(f"pickle_data/X_{img}_train_split({disp}).pickle", "wb")
            pickle.dump(X_train, pickle_out)
            pickle_out.close()

            pickle_out = open(f"pickle_data/X_{img}_val({disp}).pickle", "wb")
            pickle.dump(X_val, pickle_out)
            pickle_out.close()

    print("Divisão concluida com sucesso.")

#dividir_dados_val_only()

In [ ]:
# Gerando os outputs de extração de características da MobileNet (checkpoint)
# Somente rodar na primeira vez.
# Para evitar rodar novamente, mantendo "FE_run = False"

batch = 32
folder_i = 'pickle_data'
folder_o = 'FE_output'

FE_run = False
if FE_run:
    for disp in DISPOSITIVOS:
        for img in IMAGENS:

            # Carregamento dos dados
            X_train = pickle.load(open(f"{folder_i}/X_{img}_train({disp}).pickle","rb"))
            X_val = pickle.load(open(f"{folder_i}/X_{img}_val({disp}).pickle","rb"))
            X_test = pickle.load(open(f"{folder_i}/X_{img}_test({disp}).pickle","rb"))

            input_shape = X_train.shape[1:]
            mobilenet = MobileNetV3Large(input_shape = input_shape, weights='imagenet', include_top=False)

            for layer in mobilenet.layers:
                layer.trainable = False

            global_average_pooling_output = GlobalAveragePooling2D()(mobilenet.output)

            FE = Model(inputs=mobilenet.input, outputs=global_average_pooling_output)

            X_train_features = FE.predict(X_train, batch_size=batch, verbose=0)
            X_val_features   = FE.predict(X_val, batch_size=batch, verbose=0)
            X_test_features  = FE.predict(X_test, batch_size=batch, verbose=0)

            with open(f"{folder_o}/XFE_{img}_train({disp}).pickle", 'wb') as f:
                pickle.dump(X_train_features, f)

            with open(f"{folder_o}/XFE_{img}_val({disp}).pickle", 'wb') as f:
                pickle.dump(X_val_features, f)

            with open(f"{folder_o}/XFE_{img}_test({disp}).pickle", 'wb') as f:
                pickle.dump(X_test_features, f)
